# 10. 텍스트 임베딩과 분류 실습

- 목표: 텍스트 벡터를 실제 분류 문제에 연결하고, Word2Vec과 현대 NLP 임베딩 흐름을 개념 수준에서 정리합니다.
- 흐름: 전처리 -> 벡터화 -> 임베딩 -> 텍스트 분류 베이스라인의 전체 흐름을 복습하는 마지막 차시입니다.


## 1. 분산 표현: Word2Vec

임베딩 레이어는 **딥러닝 모델 안에 포함된 임베딩 테이블**이었습니다.  
이번에는 임베딩만 따로 학습하는 대표적인 신경망 기법인 **Word2Vec**을 살펴봅니다.


#### 1) 분산 표현(Distributed Representation)이란?

기존 통계 기반 벡터화(BoW, TF-IDF)는

- 단어마다 “등장 횟수/빈도/조합”을 세어 벡터로 만들었습니다.
- 단어 의미나 문맥을 직접 학습한다기보다, **통계를 정리한 결과**에 가깝습니다.

반면, 분산 표현은

> “비슷한 문맥에서 쓰이는 단어는 비슷한 벡터를 갖도록”  
> 신경망으로 **벡터 자체를 학습**하는 접근입니다.  
> (의미가 여러 차원에 분산되어 있다는 뜻. 원핫은 오직 한 차원('1'이 있는 그 위치)에만 정보가 있음)

- 각 단어는 보통 수십~수백 차원의 **밀집 벡터(dense vector)** 로 표현되고,
- 벡터 공간에서의 위치/거리/각도가 의미 정보를 반영하도록 학습됩니다.

#### 2) Word2Vec

Word2Vec은 **아주 얕은 신경망**을 사용해 단어 임베딩을 학습하는 모델입니다.

핵심 아이디어:

- 단어는 **주변 단어(문맥)** 과 함께 나타납니다.
- “비슷한 문맥에서 등장하는 단어들끼리는 의미가 비슷할 가능성이 크다.”
- 따라서, 문맥 예시들을 많이 보여주면서  
  임베딩이 그 패턴을 잘 예측하도록 학습합니다.

주요 구조는 두 가지입니다. 먼저 큰 방향을 보면, CBOW와 Skip-Gram은 **예측 방향**이 서로 반대입니다.

<img src="image/word2vec_cbow_skipgram_flow.svg" width="760">

이미지 출처: 수업자료용 직접 제작

아래 그림은 같은 내용을 신경망 구조에 조금 더 가깝게 보여줍니다.

<table>
  <tr>
    <td align="center"><img src="image/word2vec_cbow_commons.svg" width="360"></td>
    <td align="center"><img src="image/word2vec_skipgram_commons.svg" width="360"></td>
  </tr>
  <tr>
    <td align="center"><b>CBOW</b>: 주변 단어로 중심 단어 예측</td>
    <td align="center"><b>Skip-Gram</b>: 중심 단어로 주변 단어 예측</td>
  </tr>
</table>

외부 이미지 출처: Wikimedia Commons, [Continuous Bag of Words model (CBOW).svg](https://commons.wikimedia.org/wiki/File:Continuous_Bag_of_Words_model_(CBOW).svg), [Skip-gram.svg](https://commons.wikimedia.org/wiki/File:Skip-gram.svg) / 저자: Zhang, Aston; Lipton, Zachary C.; Li, Mu; Smola, Alexander J. / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 SVG를 파일명만 바꿔 사용

- **CBOW (Continuous Bag of Words)**  
  - 주변 단어들(컨텍스트) → 중심 단어를 예측  
  - 예: `"나는 ___을 먹었다"` → 빈칸에 들어갈 단어를 맞히도록 학습
- **Skip-Gram**  
  - 중심 단어 → 주변 단어들을 예측  
  - 예: `"밥"` → (나는, 먹었다) 같은 주변 단어들을 맞히도록 학습

학습이 잘 되면, 임베딩 벡터 공간에서  
재미있는 **벡터 연산**이 가능해지는 것으로 유명합니다.

```text
벡터("왕") - 벡터("남자") + 벡터("여자") ≈ 벡터("여왕")
```

<img src="image/word_vector_illustration_commons.jpg" width="460">

외부 이미지 출처: Wikimedia Commons, [Word vector illustration.jpg](https://commons.wikimedia.org/wiki/File:Word_vector_illustration.jpg) / 저자: Singerep / 라이선스: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) / 원본 이미지를 파일명만 바꿔 사용

즉, 단어들 사이의 관계(성별, 직업, 국가 등)가  
벡터 공간에서 **산술 연산**으로도 어느 정도 드러납니다. 


#### 3) Word2Vec vs 임베딩 레이어

Word2Vec과 임베딩 레이어는 모두 단어를 의미 벡터로 표현하지만, 학습되는 위치가 다릅니다.

| 구분 | Word2Vec | 임베딩 레이어 |
|------|----------|---------------|
| 학습 위치 | 임베딩만 따로 사전 학습 | 분류·번역 같은 모델 안에서 함께 학습 |
| 학습 신호 | 주변 단어 예측 | 최종 태스크의 정답 |
| 활용 방식 | 학습된 단어 벡터를 다른 모델에 재사용 | 특정 모델에 맞게 계속 조정 |

실제 프로젝트에서는 미리 학습된 Word2Vec 벡터를 임베딩 레이어의 초기값으로 불러와 사용하는 방식도 있습니다.


## 2. 현대 NLP에서의 임베딩: "고정된 벡터"에서 "계산되는 표현"으로

우리가 지금까지 배운 Word2Vec은 단어마다 하나의 고유한 벡터 좌표를 할당하는 **정적(Static) 임베딩**이었습니다.  
하지만 현대 NLP는 여기서 한 단계 더 진보하여, 단어의 벡터를 문장 속에서 실시간으로 만들어내는 **문맥적 임베딩(Contextual Embedding)** 시대로 넘어왔습니다.

<img src="image/static_vs_contextual_embedding.svg" width="760">

이미지 출처: 수업자료용 직접 제작

#### 1) 작동 방식의 변화: "찾아보기"에서 "실시간 합성"으로

가장 큰 차이는 임베딩이 결정되는 **시점**과 **방식**에 있습니다.

- **기존 방식 (Word2Vec):** - 메모리에 저장된 **임베딩 표(Lookup Table)** 에서 단어에 맞는 벡터를 꺼내오면 끝입니다.
- 예를 들어 "은행"은 어떤 문장에서도 항상 `[0.1, -0.5]` 같은 고정된 값입니다.
- **현대 방식 (Transformer/BERT 등):** - 임베딩 레이어에서 단어의 기본 벡터를 가져온 뒤,  
**신경망 층(Encoder)** 을 통과하며 주변 단어들의 정보를 수치적으로 **합산(Combine)** 합니다.
- 예를 들어, `"사과를 먹다"`라는 문장에서 '사과' 벡터는 옆에 있는 '먹다'라는 단어의 정보를 흡수하여  
**음식**의 특징이 실시간으로 강화된 새로운 벡터로 변신합니다.


#### 2) 학습 메커니즘의 진화: 빈칸 채우기(Masking)를 통한 문맥 파악

현대 모델은 단순히 단어의 짝을 맞히는 것을 넘어, 문장 전체의 구조를 이해하기 위해 훨씬 고차원적인 방식으로 사전 학습(Pre-training)됩니다.

- **마스크 학습 (Masked Language Modeling):**
  - 문장 중간의 단어를 가려놓고(`[MASK]`), 주변 단어들을 조합해 그 빈칸에 들어갈 단어를 맞히는 연습을 합니다.
  - 예: `"나는 오늘 [MASK]에 가서 돈을 입금했다"` → 주변의 '돈', '입금'을 보고 `[MASK]`가 '은행'임을 추론해야 합니다.
- **문맥 정보의 내재화:** 
  - 이 빈칸을 맞히기 위해 모델은 **"주변에 어떤 단어가 올 때 이 단어의 의미가 어떻게 변하는지"** 그 관계(Attention)를 아주 정교하게 학습하게 됩니다.
  - 이 과정이 수억 개의 문장에 대해 반복되면서, 단어 벡터는 단순한 의미를 넘어 **문맥을 읽어내는 능력**을 갖추게 됩니다.

#### 3) 정적 임베딩 vs 문맥적 임베딩 비교 요약

| 구분 | 정적 임베딩 (Word2Vec) | 문맥적 임베딩 (BERT, GPT 등) |
| :--- | :--- | :--- |
| **핵심 원리** | 미리 저장된 벡터를 **꺼내오기(Lookup)** | 주변 단어와 정보를 **섞어서 계산(Compute)** |
| **다의어 처리** | "은행"의 모든 의미가 하나의 벡터에 섞임 | 문맥(주변 단어)을 보고 실시간으로 의미 분리 |
| **학습 방식** | 단어 간의 단순 통계적 유사성 학습 | **빈칸 채우기(Masking)** 등으로 문맥 간 관계 학습 |
| **구성 요소** | 단어의 고유 의미 | **토큰 의미 + 순서(Position) + 문맥 정보** |

**정리:** 현대 NLP 임베딩의 핵심은 **학습 시점에 빈칸 채우기 등을 통해 단어들 사이의 고차원적인 관계를 미리 파악하고,  
이를 바탕으로 실제 문장에서 단어의 의미를 실시간으로 계산해낸다**는 점에 있습니다.

이러한 '실시간 계산 엔진'의 정체인 **Transformer**와 **Attention**의 구조를 다음 장에서 본격적으로 파헤쳐 보겠습니다.


#### 정리
- **원-핫 인코딩**: 단어 구분만 가능, 유사성 반영 불가  
- **통계 기반 임베딩**: 빈도·중요도·순서를 반영하지만, 의미는 여전히 부족  
- **분산 표현(Word2Vec)**: 신경망을 통해 **단어 간 의미 관계**까지 수치화  
- → 이는 이후 **더 복잡한 딥러닝 NLP 모델**(예: RNN, Transformer)의 기초 표현으로 활용됨 


#### 문제 1. Word2Vec 학습
문장 리스트 `["나는 학교에 간다", "학교에서 공부한다", "나는 밥을 먹었다"]`로 Word2Vec 모델을 학습한 뒤, `"학교"`와 가장 유사한 단어를 찾아보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [  # 임베딩 학습에 사용할 문장 목록입니다.
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=0)  # Word2Vec 임베딩 모델입니다. vector_size는 벡터 크기입니다.

print("학교와 가장 유사한 단어:", model.wv.most_similar("학교", topn=1))  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 작성하세요
from gensim.models import Word2Vec  # Word2Vec 임베딩 모델입니다.

sentences = [
    ["나는", "학교에", "간다"],
    ["학교에서", "공부한다"],
    ["나는", "밥을", "먹었다"],
    ["좋은", "학교", "수업"],
    ["맛있는", "밥"],
    ["시원한", "콜라를","마셨다"],
]

#### 문제 2. Word2Vec 벡터 연산
Word2Vec을 학습한 모델에서 `"밥"` - `"먹었다"` + `"마셨다"` ≈ ? 를 계산해보세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

print(model.wv.most_similar(positive=["밥", "마셨다"], negative=["먹었다"], topn=1))  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요


#### 문제 3. Word2Vec 학습 방식 바꾸기

같은 문장 리스트로 `sg=1`을 지정해 Skip-Gram 방식의 Word2Vec 모델을 학습해 보세요. 학습한 뒤 `"밥"`과 가장 유사한 단어를 확인하세요.

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

skipgram_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, sg=1)  # sg=1은 Skip-Gram 방식입니다.

print("밥과 가장 유사한 단어:", skipgram_model.wv.most_similar("밥", topn=1))  # 결과를 화면에 출력합니다.
```
</details>


In [ ]:
# 여기에 작성하세요


## 3. 웹 크롤링으로 텍스트 데이터 수집

텍스트 분류나 키워드 분석은 수집한 텍스트가 있어야 시작됩니다. 여기서는 네이버 금융에서 종목 코드를 이용해 일별 시세와 종목 뉴스를 가져와 봅니다.

네이버 금융의 일별 시세 페이지는 HTML 표 안에 데이터가 들어 있기 때문에, 실제 브라우저를 띄우는 `Selenium`이나 `Playwright`보다 가벼운 `BeautifulSoup`이 적합합니다.

아래 코드는 개발자도구의 Network 탭에서 확인할 수 있는 문서 요청 URL을 `requests.get()`으로 보내고, Elements 탭에서 확인한 표 구조를 `BeautifulSoup` 선택자로 읽는 방식입니다.


In [ ]:
import time  # 요청 사이에 잠깐 쉬기 위한 표준 라이브러리입니다.
import requests  # 웹 페이지에 HTTP 요청을 보내는 라이브러리입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
from bs4 import BeautifulSoup  # HTML에서 원하는 태그를 찾는 파서입니다.

BASE_URL = "https://finance.naver.com/item/sise_day.naver"  # 네이버 금융 일별 시세 URL입니다.
HEADERS = {  # 기본 파이썬 요청이 차단되는 경우를 줄이기 위한 요청 정보입니다.
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://finance.naver.com/",
}


def parse_price_page(code, page):
    params = {"code": code, "page": page}  # 종목 코드와 페이지 번호를 쿼리 파라미터로 보냅니다.
    response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)  # HTML 페이지를 요청합니다.
    response.raise_for_status()  # 요청 실패 시 에러를 발생시킵니다.
    response.encoding = "euc-kr"  # 네이버 금융 페이지의 한글 인코딩을 맞춥니다.

    soup = BeautifulSoup(response.text, "html.parser")  # HTML 문자열을 파싱 가능한 객체로 바꿉니다.
    rows = []  # 한 행씩 모을 리스트입니다.

    for tr in soup.select("table.type2 tr"):  # 일별 시세 표의 행을 순회합니다.
        cols = [td.get_text(" ", strip=True) for td in tr.select("td")]  # 각 칸의 텍스트만 추출합니다.
        if len(cols) != 7 or not cols[0]:  # 날짜가 없는 헤더/빈 행은 건너뜁니다.
            continue

        rows.append({  # 표 한 줄을 딕셔너리로 저장합니다.
            "date": cols[0],
            "close": cols[1],
            "change": cols[2],
            "open": cols[3],
            "high": cols[4],
            "low": cols[5],
            "volume": cols[6],
        })

    return rows


stock_code = "005930"  # 삼성전자 종목 코드입니다.
all_rows = []  # 여러 페이지의 결과를 한곳에 모읍니다.

for page in range(1, 6):  # 1~5페이지를 수집합니다.
    all_rows.extend(parse_price_page(stock_code, page))  # 현재 페이지 결과를 누적합니다.
    time.sleep(0.5)  # 서버에 부담을 주지 않도록 잠깐 쉽니다.

price_df = pd.DataFrame(all_rows)  # 수집 결과를 데이터프레임으로 바꿉니다.

numeric_cols = ["close", "change", "open", "high", "low", "volume"]  # 숫자로 바꿀 열입니다.
for col in numeric_cols:
    price_df[col] = pd.to_numeric(price_df[col].str.replace(",", "", regex=False), errors="coerce")  # 쉼표 제거 후 숫자로 변환합니다.

price_df.head()


위 코드는 종목 코드 `005930`의 일별 시세 5페이지를 가져옵니다. 종목 코드는 네이버 금융의 종목 페이지 URL에서 확인할 수 있습니다.

- `page` 값을 바꾸면 더 오래된 시세를 가져올 수 있습니다.
- `User-Agent`는 요청이 브라우저에서 온 것처럼 알려주는 역할을 합니다.
- `time.sleep(0.5)`처럼 요청 간격을 두면 서버 부담을 줄일 수 있습니다.


### 문제. 네이버 금융 종목 뉴스 수집

같은 종목 코드로 네이버 금융의 종목 뉴스에서 제목, 언론사, 날짜, 링크를 수집해 `news_df`로 만드세요.

힌트:
- URL: `https://finance.naver.com/item/news_news.naver`
- 파라미터: `code`, `page`, `sm="title_entity_id.basic"`, `clusterId=""`
- 제목: `td.title a.tit`
- 언론사: `td.info`
- 날짜: `td.date`

<details>
<summary>정답 보기</summary>

```python
from urllib.parse import urljoin  # 상대 경로 링크를 전체 URL로 바꾸는 도구입니다.

NEWS_URL = "https://finance.naver.com/item/news_news.naver"  # 네이버 금융 종목 뉴스 URL입니다.


def collect_stock_news(code, page=1):
    params = {  # 뉴스 페이지 요청에 필요한 쿼리 파라미터입니다.
        "code": code,
        "page": page,
        "sm": "title_entity_id.basic",
        "clusterId": "",
    }
    response = requests.get(NEWS_URL, params=params, headers=HEADERS, timeout=10)  # 뉴스 목록 HTML을 요청합니다.
    response.raise_for_status()  # 요청 실패 시 에러를 발생시킵니다.
    response.encoding = "euc-kr"  # 한글이 깨지지 않도록 인코딩을 맞춥니다.

    soup = BeautifulSoup(response.text, "html.parser")  # HTML 문자열을 파싱합니다.
    news_rows = []  # 뉴스 한 건씩 모을 리스트입니다.

    for tr in soup.select("table.type5 tr"):  # 뉴스 표의 행을 순회합니다.
        title_tag = tr.select_one("td.title a.tit")  # 뉴스 제목 링크 태그를 찾습니다.
        if title_tag is None:  # 제목이 없는 행은 건너뜁니다.
            continue

        source_tag = tr.select_one("td.info")  # 언론사 태그입니다.
        date_tag = tr.select_one("td.date")  # 날짜 태그입니다.

        news_rows.append({  # 뉴스 한 건을 딕셔너리로 저장합니다.
            "title": title_tag.get_text(" ", strip=True),
            "source": source_tag.get_text(" ", strip=True) if source_tag else None,
            "date": date_tag.get_text(" ", strip=True) if date_tag else None,
            "link": urljoin("https://finance.naver.com", title_tag.get("href")),
        })

    return news_rows


news_rows = collect_stock_news("005930", page=1)  # 삼성전자 뉴스 1페이지를 수집합니다.
news_df = pd.DataFrame(news_rows)  # 수집 결과를 데이터프레임으로 바꿉니다.
news_df.head()
```

</details>


수집한 `news_df["title"]`은 8번 파일의 전처리와 토큰화, 9번 파일의 TF-IDF와 임베딩, 이어지는 텍스트 분류 실습으로 자연스럽게 연결할 수 있습니다.


## 4. 텍스트 분류 미니 프로젝트

짧은 리뷰 문장을 벡터화하고, 고전적인 머신러닝 모델로 감성 분류 베이스라인을 만들어 봅니다.

- 목표: `전처리 -> 벡터화 -> 학습 -> 해석` 흐름을 한 번에 연결합니다.
- 사용 도구: `CountVectorizer`, `TfidfVectorizer`, `LogisticRegression`

<img src="image/text_classification_pipeline.svg" width="760">

이미지 출처: 수업자료용 직접 제작


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.

reviews = [
    ("배송이 빨라서 만족합니다", 1),
    ("포장이 깔끔하고 품질도 좋아요", 1),
    ("생각보다 성능이 좋아서 추천합니다", 1),
    ("가격 대비 괜찮은 선택이었어요", 1),
    ("다시 사고 싶지 않을 만큼 아쉬워요", 0),
    ("설명과 달라서 많이 실망했습니다", 0),
    ("배송이 늦고 응대도 아쉬웠어요", 0),
    ("품질이 기대보다 떨어집니다", 0),
]

df = pd.DataFrame(reviews, columns=["text", "label"])  # 데이터프레임을 직접 만듭니다.
df


### 문제 4. CountVectorizer로 문장 행렬 만들기

위의 `df["text"]`를 `CountVectorizer`로 변환하고, 단어 목록과 문서-단어 행렬을 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import CountVectorizer  # 문장을 단어 개수 벡터로 바꾸는 도구입니다.

count_vectorizer = CountVectorizer()  # 단어 빈도 벡터화 객체입니다.
X_count = count_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(count_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_count.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer  # 단어 빈도 벡터화 도구입니다.

# 여기에 작성하세요


### 문제 5. TfidfVectorizer로 다시 표현하기

같은 데이터를 `TfidfVectorizer`로 바꾸고, `CountVectorizer`와 어떤 차이가 보이는지 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.feature_extraction.text import TfidfVectorizer  # 문장을 TF-IDF 벡터로 바꾸는 도구입니다.

tfidf_vectorizer = TfidfVectorizer()  # TF-IDF 벡터화 객체입니다.
X_tfidf = tfidf_vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.

print(tfidf_vectorizer.get_feature_names_out())  # 결과를 화면에 출력합니다.
print(X_tfidf.toarray())  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 도구입니다.

# 여기에 작성하세요


### 문제 6. 로지스틱 회귀로 감성 분류 베이스라인 만들기

`TfidfVectorizer`와 `LogisticRegression`을 이용해 간단한 감성 분류 베이스라인을 만들어 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델입니다.
from sklearn.metrics import accuracy_score  # 정확도를 계산하는 함수입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터를 나누는 함수입니다.
from sklearn.pipeline import make_pipeline  # 파이프라인 생성 함수입니다.

X_train, X_test, y_train, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    df["text"], df["label"], test_size=0.25, random_state=42  # 새 파생 변수를 만들어 저장합니다.
)

model = make_pipeline(  # 전처리와 모델 파이프라인을 만듭니다.
    TfidfVectorizer(),  # TF-IDF 벡터화 객체입니다.
    LogisticRegression(max_iter=1000, random_state=42),  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
)

model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
pred = model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("accuracy:", accuracy_score(y_test, pred))  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.pipeline import make_pipeline  # 파이프라인 생성 함수입니다.

# 여기에 작성하세요


### 문제 7. 분류 결과에 영향을 많이 준 단어 확인하기

학습된 로지스틱 회귀 모델에서 긍정/부정 예측에 큰 영향을 준 단어를 확인해 보세요.

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

vectorizer = TfidfVectorizer()  # 문장을 숫자 벡터로 바꾸는 객체입니다.
X = vectorizer.fit_transform(df["text"])  # 학습과 변환을 한 번에 수행합니다.
y = df["label"]  # 정답 값 또는 두 번째 배열을 준비합니다.

clf = LogisticRegression(max_iter=1000, random_state=42)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
clf.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

feature_names = vectorizer.get_feature_names_out()  # 벡터화된 단어 목록을 가져옵니다.
coef = clf.coef_[0]  # 각 단어의 분류 가중치를 가져옵니다.

top_positive = coef.argsort()[-5:][::-1]  # 긍정 방향 가중치가 큰 단어 인덱스입니다.
top_negative = coef.argsort()[:5]  # 부정 방향 가중치가 큰 단어 인덱스입니다.

print("긍정에 가까운 단어:", feature_names[top_positive])  # 결과를 화면에 출력합니다.
print("부정에 가까운 단어:", feature_names[top_negative])  # 결과를 화면에 출력합니다.

```
</details>


In [ ]:
# 여기에 작성하세요


## 체크포인트

- 텍스트 전처리 방법은 데이터 목적에 따라 달라집니다.
- BoW, TF-IDF, Word2Vec은 모두 텍스트를 수치화하는 방법이지만 표현 방식과 장단점이 다릅니다.
- 딥러닝 모델을 쓰지 않아도 `벡터화 + 선형 모델`만으로 텍스트 분류의 기본 흐름을 만들 수 있습니다.
